In [ ]:
import json
import requests
import pandas as pd
import time
from datetime import datetime
from pathlib import Path

BASE_URL = "https://play.limitlesstcg.com/api/tournaments"
TOP_LIMIT = 2
TIMEOUT = 100
EXPORT_DIR = Path("exports")


def is_rate_limited(exc):
    """True when the exception carries a 429 Too Many Requests response."""
    response = getattr(exc, "response", None)
    return response is not None and response.status_code == 429


def _flatten_nested(dataframe):
    """Serialize nested values (lists/dicts) so the Excel writer accepts them."""
    out = dataframe.copy()
    nested = (list, dict, set, tuple)
    for col in out.columns:
        if out[col].map(lambda v: isinstance(v, nested)).any():
            out[col] = out[col].map(
                lambda v: json.dumps(v, default=str) if isinstance(v, nested) else v
            )
    return out


def export_df(dataframe, name, export_dir=EXPORT_DIR):
    """Write the DataFrame to <name>_<YYYY-MM-DD>_<HHMMSS>.csv and .xlsx."""
    export_dir.mkdir(parents=True, exist_ok=True)
    stamp = datetime.now().strftime("%Y-%m-%d_%H%M%S")
    csv_path = export_dir / f"{name}_{stamp}.csv"
    xlsx_path = export_dir / f"{name}_{stamp}.xlsx"

    flat = _flatten_nested(dataframe)
    flat.to_csv(csv_path, index=False)
    flat.to_excel(xlsx_path, index=False)

    print(f"Exported {name} ({len(dataframe)} rows) -> {csv_path} | {xlsx_path}")
    return csv_path, xlsx_path


all_tournaments = []
rate_limited = False

for page in range(1, 2):
    print(f"Fetching page {page}")

    r = requests.get(
        BASE_URL,
        params={
            "page": page,
            "game": "PTCG",
            "format": "STANDARD",
        },
        timeout=TIMEOUT,
    )

    try:
        r.raise_for_status()
    except requests.exceptions.HTTPError as e:
        if is_rate_limited(e):
            print(f"429 Client Error on page {page} - stopping iteration")
            rate_limited = True
            break
        raise

    tournaments = r.json()

    if not tournaments:
        break

    all_tournaments.extend(tournaments)

    time.sleep(TIMEOUT/1000)

df = pd.DataFrame(all_tournaments)
export_df(df, "df")

In [ ]:
df = df.drop(columns=['game'])
df['date'] = pd.to_datetime(df['date']).dt.strftime('%Y-%m-%d')
export_df(df, "df_clean")
df.head()

In [ ]:
tournament_details = []
details_rate_limited = False

# Iterating through the IDs in the existing df
for tournament_id in df['id']:
    print(f"Fetching details for: {tournament_id}")
    detail_url = f"https://play.limitlesstcg.com/api/tournaments/{tournament_id}/details"

    try:
        response = requests.get(detail_url, timeout=TIMEOUT)
        response.raise_for_status()
        tournament_details.append(response.json())
    except Exception as e:
        if is_rate_limited(e):
            print(f"429 Client Error on {tournament_id} - stopping iteration")
            details_rate_limited = True
            break
        print(f"Failed to fetch {tournament_id}: {e}")

    # Small sleep to respect the API rate limits
    time.sleep(TIMEOUT/1000)

# Create the new details DataFrame
details_df = pd.DataFrame(tournament_details)
export_df(details_df, "details_df")

In [ ]:
details_df = details_df.drop(columns=['game'])
details_df['date'] = pd.to_datetime(details_df['date'], utc=True).dt.strftime('%Y-%m-%d')
export_df(details_df, "details_df_clean")
details_df.head()

In [ ]:
# One row per tournament phase, with the phase fields as columns
phases_df = details_df.explode('phases', ignore_index=True)

phase_cols = pd.json_normalize(
    phases_df['phases'].map(lambda v: v if isinstance(v, dict) else {})
).reindex(columns=['phase', 'type', 'rounds', 'mode'])

phases_df = phases_df.drop(columns=['phases']).join(phase_cols)

export_df(phases_df, "phases_df")
phases_df.head()

In [ ]:
pairings = []
pairings_rate_limited = False

# One request per tournament, flattening every match into its own row
for tournament_id in df['id']:
    print(f"Fetching pairings for: {tournament_id}")
    pairings_url = f"https://play.limitlesstcg.com/api/tournaments/{tournament_id}/pairings"

    try:
        response = requests.get(pairings_url, timeout=TIMEOUT)
        response.raise_for_status()
        for match in response.json():
            pairings.append({"tournamentId": tournament_id, **match})
    except Exception as e:
        if is_rate_limited(e):
            print(f"429 Client Error on {tournament_id} - stopping iteration")
            pairings_rate_limited = True
            break
        print(f"Failed to fetch {tournament_id}: {e}")

    # Small sleep to respect the API rate limits
    time.sleep(TIMEOUT/1000)

# player2 is absent on byes, so make sure the column exists either way
pairings_df = pd.DataFrame(pairings).reindex(
    columns=["tournamentId", "phase", "round", "table", "player1", "player2", "winner"]
)
pairings_df["isBye"] = pairings_df["player2"].isna()

export_df(pairings_df, "pairings_df")
pairings_df.head()

In [ ]:
standings = []
standings_rate_limited = False

# One request per tournament, one row per player with deck/record flattened out
for tournament_id in df['id']:
    print(f"Fetching standings for: {tournament_id}")
    standings_url = f"https://play.limitlesstcg.com/api/tournaments/{tournament_id}/standings"

    try:
        response = requests.get(standings_url, timeout=TIMEOUT)
        response.raise_for_status()
        for entry in response.json():
            deck = entry.get("deck") or {}
            record = entry.get("record") or {}
            standings.append({
                "tournamentId": tournament_id,
                "placing": entry.get("placing"),
                "player": entry.get("player"),
                "name": entry.get("name"),
                "country": entry.get("country"),
                "deckId": deck.get("id"),
                "deckName": deck.get("name"),
                "deckIcons": deck.get("icons"),
                "wins": record.get("wins"),
                "losses": record.get("losses"),
                "ties": record.get("ties"),
                "drop": entry.get("drop"),
                "decklist": entry.get("decklist"),
            })
    except Exception as e:
        if is_rate_limited(e):
            print(f"429 Client Error on {tournament_id} - stopping iteration")
            standings_rate_limited = True
            break
        print(f"Failed to fetch {tournament_id}: {e}")

    # Small sleep to respect the API rate limits
    time.sleep(TIMEOUT/1000)

standings_df = pd.DataFrame(standings)

export_df(standings_df, "standings_df")
standings_df.head()

In [ ]:
# One row per card in every decklist, tagged with its category (pokemon/trainer/energy)
cards = []

for row in standings_df.itertuples(index=False):
    decklist = row.decklist if isinstance(row.decklist, dict) else {}
    for category, entries in decklist.items():
        for card in entries or []:
            cards.append({
                "tournamentId": row.tournamentId,
                "player": row.player,
                "placing": row.placing,
                "deckId": row.deckId,
                "category": category,
                "count": card.get("count"),
                "name": card.get("name"),
                "set": card.get("set"),
                "number": card.get("number"),
            })

decklist_cards_df = pd.DataFrame(cards)

export_df(decklist_cards_df, "decklist_cards_df")
decklist_cards_df.head()